In [ ]:
import ast
import re
import subprocess
import sys
import os


CORRECTNESS_WEIGHT = 0.7
STYLE_WEIGHT = 0.2
SYNTAX_WEIGHT = 0.1
FINALIZATION_THRESHOLD = 90


class PythonCodeAnalyzer:
    def __init__(self, code, file_path):
        """
        Initialize the analyzer with the given code and file path.

        Args:
            code (str): The Python code to analyze.
            file_path (str): Path to the Python script file.
        """
        self.code = code
        self.file_path = file_path
        self.tree = None
        self.score = 100  # Start with a perfect score
        self.dependencies = []  # List of dependencies (files/modules)

    def parse_code(self):
        """
        Parses the Python code and checks for syntax errors.

        Returns:
            bool: True if parsing is successful, False otherwise.
        """
        try:
            self.tree = ast.parse(self.code)
            print("Code parsed successfully.")
        except SyntaxError as e:
            print(f"Syntax Error: {e}")
            self.score -= 50  # Penalize for syntax errors
            return False
        return True

    def analyze_dependencies(self):
        """
        Analyzes the code for external dependencies (imported files/modules).
        """
        print("Analyzing dependencies...")
        for node in ast.walk(self.tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    self.dependencies.append(alias.name)
            elif isinstance(node, ast.ImportFrom):
                self.dependencies.append(node.module)

        missing_dependencies = []
        for dep in self.dependencies:
            if not self.is_dependency_available(dep):
                missing_dependencies.append(dep)

        if missing_dependencies:
            print(f"Missing dependencies: {missing_dependencies}")
            self.score -= len(missing_dependencies) * 5  # Penalize for missing dependencies
        else:
            print("All dependencies are available.")

    def is_dependency_available(self, dep):
        """
        Checks if a dependency is available.

        Args:
            dep (str): The dependency name.

        Returns:
            bool: True if available, False otherwise.
        """
        try:
            __import__(dep)
            return True
        except ImportError:
            return os.path.isfile(dep)  # Check for local file if not a module

    def execute_and_capture_output(self):
        """
        Executes the input script and captures its output.

        Returns:
            tuple: (success, output/error message)
        """
        print("Executing the script to capture output...")
        try:
            result = subprocess.run(
                [sys.executable, self.file_path],
                text=True,
                capture_output=True,
                check=True
            )
            print("Execution output:")
            print(result.stdout)
            return True, result.stdout
        except subprocess.CalledProcessError as e:
            print(f"Error during execution: {e.stderr}")
            return False, e.stderr

    def analyze_functions(self):
        """
        Analyzes the functions in the code for adherence to good practices.
        """
        if not self.tree:
            print("No code to analyze.")
            return

        print("Analyzing functions...")
        function_count = 0
        for node in ast.walk(self.tree):
            if isinstance(node, ast.FunctionDef):
                function_count += 1
                print(f"Function found: {node.name}")
                print(f"Arguments: {[arg.arg for arg in node.args.args]}")
                print(f"Body length: {len(node.body)} lines")

        if function_count == 0:
            print("No functions found in the code.")
            self.score -= 20  # Penalize if no functions are defined

    def check_unused_imports(self):
        """
        Checks for unused imports in the code.
        """
        if not self.tree:
            print("No code to analyze.")
            return

        print("Checking for unused imports...")
        imports = [node for node in ast.walk(self.tree) if isinstance(node, ast.Import)]
        if imports:
            print("Imports found:")
            for imp in imports:
                print(f" - {imp.names[0].name}")
        else:
            print("No imports found.")

    def calculate_score(self):
        """
        Displays the final analysis score.
        """
        print(f"Final Score: {self.score}")


def read_file(file_path):
    """
    Reads the content of a Python file.

    Args:
        file_path (str): The path to the Python file.

    Returns:
        str: The content of the file or None if file is not found.
    """
    if not os.path.isfile(file_path):
        print(f"File not found: {file_path}")
        return None

    try:
        with open(file_path, 'r') as file:
            return file.read()
    except Exception as e:
        print(f"Error reading file: {e}")
        return None


if __name__ == "__main__":
    # Prompt the user for the Python file path
    file_path = input("Enter the path to the Python file to analyze: ").strip()
    code = read_file(file_path)

    if code:
        analyzer = PythonCodeAnalyzer(code, file_path)
        if analyzer.parse_code():
            analyzer.analyze_dependencies()  # Check dependencies
            analyzer.analyze_functions()
            analyzer.check_unused_imports()

            # Execute the script and check output
            success, output = analyzer.execute_and_capture_output()
            if not success:
                analyzer.score -= 10  # Penalize for runtime errors
        analyzer.calculate_score()
